# Fine-Tuning Only Notebook

This notebook is only for LoRA fine-tuning, adapter testing, and adapter evaluation for the GenAI Mentor project.

Run the cells in order:
1. Set up paths and device.
2. Prepare the SFT train/validation/test splits.
3. Train a bounded LoRA adapter on MPS/CUDA.
4. Test the adapter with one prompt.
5. Evaluate the adapter on role-specific prompts.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import FINETUNE_BASE_MODEL, FINETUNE_DIR, OUTPUTS_DIR
from src.finetuning.prepare_dataset import prepare_finetune_dataset

print(f"Project root: {ROOT}")
print(f"Base model: {FINETUNE_BASE_MODEL}")
print(f"Fine-tuning data: {FINETUNE_DIR}")
print(f"Fine-tuning outputs: {OUTPUTS_DIR / 'finetune'}")

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
device = "cuda" if cuda_available else "mps" if mps_available else "cpu"

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {cuda_available}")
print(f"MPS available: {mps_available}")
print(f"Selected device: {device}")

if device == "cpu":
    raise RuntimeError("Fine-tuning needs CUDA or Apple MPS for this notebook run.")

## Prepare Fine-Tuning Splits

This creates `train.jsonl`, `val.jsonl`, and `test.jsonl` from the existing chat-format SFT dataset.

In [ ]:
def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())

prepare_finetune_dataset()

split_counts = {
    "sft_chat_dataset": count_jsonl(FINETUNE_DIR / "sft_chat_dataset.jsonl"),
    "train": count_jsonl(FINETUNE_DIR / "train.jsonl"),
    "val": count_jsonl(FINETUNE_DIR / "val.jsonl"),
    "test": count_jsonl(FINETUNE_DIR / "test.jsonl"),
}
split_counts

## Train LoRA Adapter

The default values below are intentionally bounded so the run finishes on Apple MPS. Increase `FINETUNE_MAX_TRAIN_EXAMPLES` when you want a stronger adapter.

In [ ]:
os.environ["FINETUNE_MAX_TRAIN_EXAMPLES"] = "32"
os.environ["FINETUNE_MAX_EVAL_EXAMPLES"] = "8"
os.environ["FINETUNE_MAX_LENGTH"] = "512"
os.environ["FINETUNE_EPOCHS"] = "1"

from src.finetuning.train_lora import train_lora

train_lora()

training_log_path = OUTPUTS_DIR / "finetune" / "training_log.json"
training_log = json.loads(training_log_path.read_text(encoding="utf-8"))
training_log

In [ ]:
adapter_dir = OUTPUTS_DIR / "finetune" / "lora_adapter"
adapter_files = sorted(path.name for path in adapter_dir.glob("*"))

assert training_log["status"] == "completed", training_log
assert (adapter_dir / "adapter_model.safetensors").exists(), adapter_files
assert (adapter_dir / "adapter_config.json").exists(), adapter_files

print(f"Adapter directory: {adapter_dir}")
adapter_files

## Test The Adapter

This cell loads the saved LoRA adapter and generates one answer.

In [ ]:
from src.finetuning.inference_lora import generate_with_lora

test_prompt = """Mode: tutor
Instruction: Teach the concept using the required tutoring format.
Input:
Student level: beginner
Question: What is retrieval augmented generation?
"""

test_output = generate_with_lora(test_prompt, max_new_tokens=160)
print(test_output)

## Evaluate The Adapter

This lightweight evaluation checks whether the adapter follows the required role-specific response structure.

In [ ]:
import pandas as pd

eval_cases = [
    {
        "mode": "tutor",
        "prompt": """Mode: tutor
Instruction: Teach the concept using the required tutoring format.
Input:
Student level: beginner
Question: What is LoRA fine-tuning?
""",
        "required_phrases": ["Simple explanation", "Analogy", "Course-grounded answer", "Common misconception", "Quick check question"],
    },
    {
        "mode": "examiner",
        "prompt": """Mode: examiner
Instruction: Grade the student's answer using the lecture context.
Input:
Question: What is RAG?
Student answer: RAG retrains the model every time a user asks a question.
""",
        "required_phrases": ["Score", "Feedback", "Corrected answer"],
    },
    {
        "mode": "critic",
        "prompt": """Mode: critic
Instruction: Critique and improve the assistant answer using the lecture context.
Input:
Question: How do prompts affect LLM behavior?
Assistant answer: Prompts do not matter because the model already knows everything.
""",
        "required_phrases": ["Critique", "Groundedness", "Missing points", "Improved answer"],
    },
]

rows = []
for case in eval_cases:
    output = generate_with_lora(case["prompt"], max_new_tokens=180)
    found = [phrase for phrase in case["required_phrases"] if phrase.lower() in output.lower()]
    rows.append({
        "mode": case["mode"],
        "required": len(case["required_phrases"]),
        "found": len(found),
        "structure_score": round(len(found) / len(case["required_phrases"]), 2),
        "found_phrases": ", ".join(found),
        "output_preview": output[:500],
    })

eval_df = pd.DataFrame(rows)
eval_df

In [ ]:
eval_output_path = OUTPUTS_DIR / "finetune" / "notebook_adapter_eval.csv"
eval_df.to_csv(eval_output_path, index=False)
print(f"Saved adapter evaluation to: {eval_output_path}")